# سبق ۱۰ - پیداوار میں AI ایجنٹس  

اس سبق میں آپ سیکھیں گے **پیداوری پیٹرنز** AI ایجنٹس کے لیے Microsoft Agent Framework (Python) استعمال کرتے ہوئے۔ ہم شامل کرتے ہیں:  

- **مشاہدہ** — ایجنٹ کے تعاملات میں وقت لگانے اور لاگنگ شامل کرنا  
- **تشخیص** — جواب کی معیار کی جانچ کے لیے ایک جانچ ایجنٹ کا استعمال  
- **لاگت کا انتظام** — ٹوکن کی بچت اور ماڈل کے انتخاب کی حکمت عملی  

منظر نامہ ایک **ٹریول ایجنٹ** ہے جو صارفین کی مدد کرتا ہے سفر کی منصوبہ بندی میں، جس پر نگرانی اور تشخیص شامل ہے۔  


## تنصیب


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## پیداواری غور و فکر

مصنوعی ذہانت کے ایجنٹس کو پروٹوٹائپس سے پیداوار میں منتقل کرنے کے لیے تین اہم ستونوں پر محتاط توجہ درکار ہوتی ہے:

1. **مشاہدہ پذیری** — آپ کو یہ دیکھنے کی ضرورت ہوتی ہے کہ ایجنٹ کیا کر رہا ہے، اسے کتنا وقت لگتا ہے، اور وہ کون سے ٹولز کال کر رہا ہے۔ بغیر ٹریسنگ اور لاگنگ کے، پیداواری مسائل کی خرابی تلاش کرنا تقریباً ناممکن ہے۔

2. **تشخیص** — خودکار معیار کی جانچ اس بات کو یقینی بناتی ہے کہ ایجنٹ کے جوابات وقت کے ساتھ درست، مکمل، اور مددگار رہیں۔ ایک تشخیصی ایجنٹ متعین معیار کے خلاف جوابات کا اسکور کر سکتا ہے۔

3. **لاگت کا انتظام** — ٹوکن کا استعمال براہ راست لاگت کو متاثر کرتا ہے۔ پرامپٹ کی بہتری، ماڈل کا انتخاب، اور کیشنگ جیسی حکمت عملیوں سے بغیر معیار قربان کیے اخراجات کو کنٹرول میں رکھا جا سکتا ہے۔


## ایک قابل مشاہدہ ایجنٹ بنانا

ہم سفر کے اوزار کی تعریف کرتے ہیں اور ایجنٹ کال کو ٹائمنگ کے ساتھ لپیٹتے ہیں تاکہ ہم تاخیر کی نگرانی کر سکیں۔ پروڈکشن میں آپ OpenTelemetry یا کسی مماثل ٹریسنگ بیک اینڈ کے ساتھ انضمام کریں گے۔


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## جائزہ لینے کے نمونے

ایک عام پیداواری نمونہ یہ ہے کہ ایک دوسرے ایجنٹ کو **جائزہ لینے والا** کے طور پر استعمال کیا جائے۔ جائزہ لینے والا بنیادی ایجنٹ کے جواب کو پہلے سے متعین معیار جیسے کہ مکمل ہونا، درستگی، اور مددگار ہونے کے خلاف اسکور کرتا ہے۔

یہ قابلِ عمل بناتا ہے:
- صارفین تک جوابات پہنچنے سے پہلے خودکار کوالٹی گیٹ
- جب پرامپٹ یا ماڈلز تبدیل ہوں تو ریگریشن کا پتہ لگانا
- وقت کے ساتھ ایجنٹ کی کارکردگی کا مسلسل جائزہ لینا


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## لاگت مینجمنٹ کی حکمت عملیاں

پیداوار کے AI ایجنٹس کے لیے لاگت کو کنٹرول کرنا بہت اہم ہے۔ یہاں اہم حکمت عملیاں دی گئی ہیں:

| حکمت عملی | وضاحت |
|---|---|
| **پرومپٹ کی اصلاح** | نظام کی ہدایات کو مختصر رکھیں۔ ان پٹ ٹوکنز کو کم کرنے کے لیے غیر ضروری سیاق و سباق کو ہٹا دیں۔ |
    "| **ماڈل کا انتخاب** | سادہ کاموں جیسے درجہ بندی یا استخراج کے لیے چھوٹے، سستے ماڈلز (مثلاً GPT-4.1-mini) استعمال کریں، اور پیچیدہ استدلال کے لیے بڑے ماڈلز کو محفوظ رکھیں۔ |\n",
| **کیچنگ** | ٹول کے نتائج اور اکثر پوچھے جانے والے سوالات کو کیش کریں تاکہ بار بار API کالز سے بچا جا سکے۔ |
| **ٹوکن بجٹ** | غیر متوقع طویل جوابات سے بچنے کے لیے `max_tokens` کی حد مقرر کریں۔ |
| **بیچنگ** | جہاں ممکن ہو، متعدد صارف کے سوالات کو ایک API کال میں گروپ کریں۔ |

عملی طور پر، ایک درجے وار طریقہ کار اچھا کام کرتا ہے: سیدھے سادے درخواستوں کو ایک تیز، سستا ماڈل بھیجیں اور صرف پیچیدہ سوالات کو زیادہ قابل ماڈل کی طرف بڑھائیں۔


## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

1. ایجنٹ کی تعاملات میں **نظارت شامل کریں** وقت بندی اور لاگنگ کے ساتھ، جو ٹریسنگ اور مانیٹرنگ کی بنیاد فراہم کرتی ہے۔
2. ایجنٹ کے جوابات کو خودکار طریقے سے **جانچیں** ایک جائزہ لینے والے ایجنٹ کے ذریعے جو مکملیت، درستگی، اور مددگار ہونے کی درجہ بندی کرتا ہے۔
3. لاگت کو **منظم کریں** پرامپٹ کی اصلاح، ماڈل کے انتخاب، کیشنگ، اور ٹوکن بجٹس کے ذریعے۔

یہ پیداوار کے نمونے آپ کے AI ایجنٹس کو قابل اعتماد، قابل پیمائش، اور پیمانے پر لاگت موثر بنانے میں مدد دیتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
